In [31]:
import pandas as pd
import numpy as np
ES=pd.read_csv("./ES_full_5min_continuous_absolute_adjusted.txt", names=["DateTime","Open","High","Low","Close","Volume"],parse_dates=['DateTime'])
ES.set_index('DateTime',inplace=True)
ES=ES.between_time("09:30","16:00","left")


In [36]:
ES['num'] = ES.groupby(ES.index.date).cumcount() + 1

counts = ES.groupby(ES.index.date).size()

full_days_set = set(counts[counts == 78].index)
mask = [d.date() in full_days_set for d in ES.index]
ES=ES[mask]

In [74]:
ES.columns

Index(['Open', 'High', 'Low', 'Close', 'Volume', 'num'], dtype='object')

In [39]:
rth=ES.resample("1D").agg({"Open":'first','High':'max','Low':'min','Close':'last'}).dropna()

In [40]:
df=rth.copy()
#df_p=pd.DataFrame()
df['range']=(df['High']-df['Low'])
df['rth_ret']=df['Close']-df['Open']
df['C2Crth']=df['Close']-df['Close'].shift(1)
df['gap']=df['Open']-df['Close'].shift(1)

In [103]:
df2=rth.copy()
df2['range']=df['range']/df['Close'].shift(1)
df2['rth_ret']=df['rth_ret']/df['Close'].shift(1)
df2['log_ret']=np.log(df2['Close']/df2['Close'].shift(1))
df2['gap']=df['gap']/df['Close'].shift(1)

df2['Open_ratio']=(df['Open']-df['Low'])/(df['High']-df['Low'])
df2['Close_ratio']=(df['Close']-df['Low'])/(df['High']-df['Low'])

In [75]:
import numpy as np
import pandas as pd

def split_by_day_dict(df):
    dates = df.index.normalize()
    split_idx = np.flatnonzero(dates[1:] != dates[:-1]) + 1
    
    arr = df[['num','High','Low']].to_numpy()
    arr_split = np.split(arr, split_idx)
    date_split = np.split(dates.to_numpy(), split_idx)
    
    return {d[0]: a for d, a in zip(date_split, arr_split)}

def findmaxmin(df):
    df_dic = split_by_day_dict(df)

    rows = []

    for d, day in df_dic.items():
        nums  = day[:,0]
        highs = day[:,1]
        lows  = day[:,2]

        n_max_high = nums[highs.argmax()]
        n_min_low  = nums[lows.argmin()]

        rows.append([
            d,
            n_min_low,
            n_max_high,
            min(n_min_low, n_max_high),
            max(n_min_low, n_max_high)
        ])

    return pd.DataFrame(
        rows,
        columns=['Date','LoD_bar','HoD_bar','first_ex','second_ex']
    ).set_index('Date')

In [77]:
df3=findmaxmin(ES)

In [ ]:
from pathlib import Path
import pandas as pd
BASE = Path("ES_sample_chart_adjusted").resolve()

image_map = {
    pd.to_datetime(p.name[:10]): p
    for p in BASE.rglob("*.jpg")
}



In [95]:
from pathlib import Path
import webbrowser

def open_gallery(paths, cols=5, outfile="chart_gallery.html"):
    html = """
    <html>
    <head>
    <style>
    body { font-family: Arial; }
    .grid { display: grid; grid-template-columns: repeat(""" + str(cols) + """, 1fr); gap: 10px; }
    img { width: 100%; border: 1px solid #ccc; }
    .cell { text-align: center; font-size: 12px; }
    </style>
    </head>
    <body>
    <div class="grid">
    """

    for p in paths:
        html += f"""
        <div class="cell">
            <img src="{Path(p).resolve().as_uri()}">
            <div>{Path(p).name[:10]}</div>
        </div>
        """

    html += """
    </div>
    </body>
    </html>
    """

    with open(outfile, "w", encoding="utf-8") as f:
        f.write(html)

    webbrowser.open(Path(outfile).resolve().as_uri())

In [104]:
selected = df2[df2["Close_ratio"] > 0.9]
print (f'{len(selected)/len(rth)}% of days meet the criteria')

0.17199289835774523% of days meet the criteria


In [105]:
paths = [image_map[d] for d in selected.index if d in image_map]


open_gallery(paths, cols=1)